# SENTINEL — Train hành vi "đánh nhau" (luồng A: đặc trưng hình ảnh)

Pipeline: tải dataset → loại video trùng → trích đặc trưng S3D (pretrain Kinetics-400, đóng băng) cho từng giây video → train đầu dự đoán theo thời gian → đánh giá trên tập test, so với luồng B (khung xương, `fight_v3`).

**Vì sao đổi sang luồng A:** sau 3 lần train, luồng B đạt AUC theo frame 0,81 trên toàn bộ tập test nhưng AUC trung vị *trong từng video đánh nhau* chỉ 0,54. Đếm số người còn phân biệt video đánh nhau tốt hơn model. Model học "cảnh đông, khung xương rõ" chứ không học động tác.

**Thiết kế (áp dụng bài học của luồng B ngay từ đầu):**

| Thành phần | Lựa chọn | Lý do |
|---|---|---|
| Backbone | S3D (torchvision, Kinetics-400 top-1 68%), đóng băng | Họ I3D; trên CPU máy chạy thật (Ryzen 7 7735HS, 8 luồng) mất ~390 ms/clip, trích cả pipeline nhanh ~4,7 lần thời gian thực. MViT/Swin3D > 1 s/clip |
| Clip | 16 frame lấy ở 15 FPS (~1 s), mỗi giây một clip, cả khung hình resize về 224 | Không cắt mép: đánh nhau ở rìa khung vẫn thấy |
| Đặc trưng | 2048 chiều = trung bình + max theo không gian | Đám đánh nhau nhỏ trong khung hình rộng không bị trung bình hóa mất |
| Đầu dự đoán | Tích chập nhân quả 3 tầng, 15 s ngữ cảnh, LayerNorm | Chỉ nhìn quá khứ như camera trực tiếp; điểm chạy trực tiếp = điểm offline (có test) |
| Loss | BCE cân bằng hai lớp + **xếp hạng trong cùng video** (đoạn đánh phải cao hơn đoạn yên) | Chỉ học được bằng chính hành động, không bằng "video kiểu nào" |
| Lấy mẫu | 60% chuỗi 64 s từ video có đánh nhau, phần lớn chứa cả đoạn đánh và đoạn yên | Buộc phân biệt trong cùng cảnh |
| Chọn checkpoint | Trung bình AUC gộp và AUC trong video | AUC gộp một mình thưởng cho đường tắt |
| Đường cơ sở | **Năng lượng chuyển động** (sai khác giữa các frame) | Model phải vượt nó, nếu không chỉ là "phát hiện chuyển động mạnh" |

**Khối lượng:** trích đặc trưng ~80 giờ video một lần (lưu lên Drive, chạy lại bỏ qua video đã có), nút thắt là giải mã video; mục 5 in tốc độ thực tế. Train đầu dự đoán chỉ vài phút đến vài chục phút vì đặc trưng đã có sẵn. Mọi bước chạy tiếp được sau khi Colab ngắt.

**GPU:** `Runtime > Change runtime type > L4 GPU`.

**Treo máy:** giữ `AUTO_DISCONNECT = True` rồi `Runtime > Run all`. Chạy xong hoặc lỗi ở mục nào thì notebook ghi lý do vào `MyDrive/SENTINEL/artifacts/<RUN_NAME>/run_log.txt`, đồng bộ Drive rồi tự ngắt runtime. Giữ tab trình duyệt mở.

## 0. Cấu hình

In [ ]:
from pathlib import Path

REPO_URL = "https://github.com/TranDuon/anomaly-behavior-detector-system.git"
BRANCH = "main"
DRIVE_ROOT = Path("/content/drive/MyDrive/SENTINEL")
REPO_DIR = DRIVE_ROOT / "repo" / "anomaly-behavior-detector-system"
ZIP_DIR = DRIVE_ROOT / "datasets" / "fight"          # chỉ giữ file zip gốc
FEATURE_ROOT = DRIVE_ROOT / "features" / "s3d_v1"     # đặc trưng S3D (dùng lại giữa các lần train)
POSE_ROOT = DRIVE_ROOT / "features" / "pose_v1"       # khung xương của luồng B: chỉ dùng để chia nhóm đông người
RUN_NAME = "visual_fight_v1"   # mỗi lần train lại đặt tên mới để giữ kết quả cũ mà so sánh
COMPARE_RUN = "fight_v3"       # luồng B tốt nhất, để so sánh ở mục 11; "" nếu không có
ARTIFACT_DIR = DRIVE_ROOT / "artifacts" / RUN_NAME
DEDUP_DIR = DRIVE_ROOT / "artifacts" / "fight" / "dedup"   # dùng chung với luồng B (giữ cache hash)
LOCAL_DATA = Path("/content/datasets")                 # video giải nén (mất khi hết phiên)

UBI_URL = "https://socia-lab.di.ubi.pt/EventDetection/UBI_FIGHTS.zip"
UBI_ZIP_BYTES = 8_117_112_419
SCF_URL = "https://github.com/sayibet/fight-detection-surv-dataset.git"
KEEP_UBI_ZIP_ON_DRIVE = True   # False: tải lại mỗi phiên, tiết kiệm 8,1 GB Drive
USE_RLVS = True                # cần MyDrive/SENTINEL/kaggle.json (mục 4b); thiếu file thì bỏ qua RLVS và ghi vào run_log
SHORT_TEST_SECONDS = 60        # báo cáo thêm nhóm "_short" (video test bình thường cắt 60 s) để so với luồng B lần 1–2

# Trích đặc trưng (mục 7)
EXTRACT_BATCH = 16             # số clip mỗi lần chạy S3D

# Train (mục 8)
EPOCHS = 30
STEPS_PER_EPOCH = 200
BATCH_SIZE = 32                # số chuỗi mỗi batch
SEQUENCE_LENGTH = 64           # số clip (giây) mỗi chuỗi
FIGHT_SHARE = 0.6              # tỉ lệ chuỗi lấy từ video có đánh nhau
RANK_WEIGHT = 1.0              # trọng số loss xếp hạng trong video
SELECT_ON = "mean"             # chọn checkpoint: "mean" (AUC gộp + AUC trong video) / "video_auc" / "frame_auc"

AUTO_DISCONNECT = True         # treo máy: chạy xong hoặc gặp lỗi thì ghi log, đồng bộ Drive rồi ngắt runtime

UBI_ROOT = LOCAL_DATA / "UBI_FIGHTS"
SCF_ROOT = LOCAL_DATA / "fight-detection-surv-dataset"
RLVS_ROOT = LOCAL_DATA / "rlvs"
print("Feature cache:", FEATURE_ROOT)
print("Artifacts:", ARTIFACT_DIR)

## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
for path in [REPO_DIR.parent, ZIP_DIR, FEATURE_ROOT, ARTIFACT_DIR, LOCAL_DATA]:
    path.mkdir(parents=True, exist_ok=True)

import time
from contextlib import contextmanager
RUN_LOG = ARTIFACT_DIR / "run_log.txt"

def note(message):
    # Ghi một dòng vào log trên Drive (không ngắt runtime).
    with open(RUN_LOG, "a", encoding="utf-8") as log:
        log.write(f"{time.strftime('%Y-%m-%d %H:%M:%S')} {message}\n")
    print(message)

def finish_and_disconnect(message):
    # Ghi log lên Drive; khi AUTO_DISCONNECT thì đồng bộ Drive rồi ngắt runtime để dừng tốn compute unit.
    note(message)
    if AUTO_DISCONNECT:
        from google.colab import runtime
        drive.flush_and_unmount()
        runtime.unassign()

@contextmanager
def step(name):
    try:
        yield
    except Exception as error:
        finish_and_disconnect(f"LỖI ở mục {name}: {error!r} — xem output của mục này, sửa rồi chạy lại.")
        raise

print("Log chạy:", RUN_LOG, "| AUTO_DISCONNECT =", AUTO_DISCONNECT)

## 2. Clone lần đầu / đồng bộ code mới nhất
GitHub là nguồn chuẩn: cell bỏ mọi thay đổi trong file tracked trên Drive rồi đồng bộ với `origin`.

In [ ]:
with step('2. Đồng bộ code'):
    import os, subprocess

    if not (REPO_DIR / ".git").exists():
        subprocess.run(["git", "clone", "--branch", BRANCH, REPO_URL, str(REPO_DIR)], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "checkout", BRANCH], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "reset", "--hard", f"origin/{BRANCH}"], check=True)
    os.chdir(REPO_DIR)
    subprocess.run(["git", "log", "--oneline", "-1"], check=True)

## 3. Cài dependency, kiểm tra GPU, tải trọng số S3D
Tải trọng số một lần trước khi chạy song song, để các tiến trình trích đặc trưng không cùng tải một file.

In [ ]:
with step('3. Cài dependency'):
    import sys, torch
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
    CPUS = os.cpu_count() or 2
    EXTRACT_SHARDS = max(1, min(8, CPUS - 1))   # tiến trình trích đặc trưng song song (giải mã video là nút thắt)
    print("PyTorch", torch.__version__, "| device:", DEVICE, "|", torch.cuda.get_device_name(0) if DEVICE == "cuda" else "KHÔNG CÓ GPU")
    print(f"CPU: {CPUS} | extract shards: {EXTRACT_SHARDS}")
    from torchvision.models.video import S3D_Weights
    S3D_Weights.KINETICS400_V1.get_state_dict(progress=False)
    print("Trọng số S3D:", S3D_Weights.KINETICS400_V1.url)

## 4. Tải dataset

**UBI-Fights:** chứng chỉ SSL của trang tác giả đã hết hạn nên dùng `--no-check-certificate`. File zip 8,1 GB; `-c` cho phép tải tiếp khi bị ngắt.

In [ ]:
with step('4. Tải dataset'):
    ubi_zip = (ZIP_DIR if KEEP_UBI_ZIP_ON_DRIVE else Path("/content")) / "UBI_FIGHTS.zip"
    if not ubi_zip.exists() or ubi_zip.stat().st_size != UBI_ZIP_BYTES:
        subprocess.run(["wget", "-c", "--no-check-certificate", "-q", "--show-progress", UBI_URL, "-O", str(ubi_zip)], check=True)
    assert ubi_zip.stat().st_size == UBI_ZIP_BYTES, "File zip UBI-Fights chưa tải đủ"
    if not (UBI_ROOT / "test_videos.csv").exists():
        subprocess.run(["unzip", "-q", "-o", str(ubi_zip), "-d", str(LOCAL_DATA)], check=True)
    print("UBI-Fights videos:", len(list(UBI_ROOT.glob("videos/*/*.mp4"))))

    if not SCF_ROOT.exists():
        subprocess.run(["git", "clone", "--depth", "1", SCF_URL, str(SCF_ROOT)], check=True)
    print("Surveillance Camera Fight videos:", len(list(SCF_ROOT.rglob("*.mp4"))) + len(list(SCF_ROOT.rglob("*.avi"))))

### 4b. Real Life Violence Situations
Tải `kaggle.json` từ trang tài khoản Kaggle, upload vào `MyDrive/SENTINEL/kaggle.json`. Thiếu file thì notebook train không có RLVS và ghi cảnh báo vào `run_log.txt` thay vì dừng. Dataset chưa ghi giấy phép rõ ràng, tác giả yêu cầu trích dẫn; chỉ dùng cho nghiên cứu.

In [ ]:
with step('4b. RLVS'):
    import shutil
    DATASETS_TRAIN = [f"ubi:{UBI_ROOT}"]
    kaggle_json = DRIVE_ROOT / "kaggle.json"
    if USE_RLVS and not kaggle_json.exists():
        note(f"CẢNH BÁO: không có {kaggle_json} — lần train này KHÔNG dùng RLVS.")
    elif USE_RLVS:
        kaggle_dir = Path.home() / ".kaggle"; kaggle_dir.mkdir(exist_ok=True)
        shutil.copy2(kaggle_json, kaggle_dir / "kaggle.json")
        os.chmod(kaggle_dir / "kaggle.json", 0o600)
        if shutil.which("kaggle") is None:
            subprocess.run([sys.executable, "-m", "pip", "install", "-q", "kaggle"], check=True)
        if not RLVS_ROOT.exists():
            subprocess.run(["kaggle", "datasets", "download", "-d", "mohamedmustafa/real-life-violence-situations-dataset",
                            "-p", str(RLVS_ROOT), "--unzip"], check=True)
        DATASETS_TRAIN.append(f"rlvs:{RLVS_ROOT}")
    DATASETS_TEST = [f"ubi:{UBI_ROOT}", f"scf:{SCF_ROOT}"]
    DATASETS_ALL = DATASETS_TRAIN + [f"scf:{SCF_ROOT}"]
    print(DATASETS_ALL)

## 5. Smoke test
Trích đặc trưng 2 video test vào thư mục tạm, kiểm tra kích thước và tốc độ (lần thời gian thực) trước khi chạy toàn bộ.

In [ ]:
with step('5. Smoke test'):
    import json
    import numpy as np

    def run(script, *args):
        command = [sys.executable, "-u", script, *[str(a) for a in args]]
        subprocess.run(command, cwd=REPO_DIR, check=True)

    def dataset_args(specs):
        return [part for spec in specs for part in ("--dataset", spec)]

    smoke_root = Path("/content/visual_smoke")
    run("scripts/extract_visual.py", *dataset_args([f"ubi:{UBI_ROOT}"]), "--feature-root", smoke_root,
        "--split", "test", "--limit", "2", "--device", DEVICE, "--batch-size", EXTRACT_BATCH, "--force")
    videos = json.loads((smoke_root / "ubi" / "manifest.json").read_text())["videos"]
    for key, info in videos.items():
        with np.load(smoke_root / "ubi" / f"{key}.npz") as data:
            features = data["features"]
            assert features.shape[1] == 2048 and np.isfinite(features.astype(np.float32)).all()
        print(f"{key}: {info['clips']} clip, {info['frames']} frame, {info['seconds']} s")
    assert all(info["clips"] > 0 for info in videos.values()), "Không trích được clip nào"

## 6. Loại video trùng giữa các dataset
Chỉ cặp khác dataset bị loại khỏi train; dùng chung kết quả (và cache hash) với luồng B.

In [ ]:
with step('6. Loại video trùng'):
    run("scripts/dedup_videos.py", *dataset_args(DATASETS_ALL), "--output-dir", DEDUP_DIR)
    report = json.loads((DEDUP_DIR / "exclude.json").read_text())
    print("Cặp trùng khác dataset:", len(report["duplicate_pairs"]))
    print("Loại khỏi train:", report["exclude_from_training"][:20])

## 7. Trích đặc trưng S3D (song song)
`EXTRACT_SHARDS` tiến trình cùng dùng GPU (FP16), mỗi tiến trình nhận một phần danh sách video và ghi log riêng ở `/content/visual_shard*.log`. Luôn trích **toàn bộ** video. Mỗi video một file trên Drive; chạy lại cell sau khi Colab ngắt thì video đã có được bỏ qua. Video gốc không bị sửa.

In [ ]:
with step('7. Trích đặc trưng'):
    import time
    processes = []
    for shard in range(EXTRACT_SHARDS):
        log = open(f"/content/visual_shard{shard}.log", "w")
        command = [sys.executable, "-u", "scripts/extract_visual.py", *dataset_args(DATASETS_ALL),
                   "--feature-root", str(FEATURE_ROOT), "--split", "all", "--device", DEVICE,
                   "--batch-size", str(EXTRACT_BATCH), "--shard", f"{shard}/{EXTRACT_SHARDS}"]
        processes.append((subprocess.Popen(command, cwd=REPO_DIR, stdout=log, stderr=subprocess.STDOUT), log))

    def last_line(shard):
        lines = Path(f"/content/visual_shard{shard}.log").read_text(errors="ignore").strip().splitlines()
        return lines[-1] if lines else "(chưa có output)"

    started = time.time()
    while any(process.poll() is None for process, _ in processes):
        time.sleep(300)
        print(f"[{(time.time() - started) / 60:.0f} phút]")
        for shard in range(EXTRACT_SHARDS):
            print(f"   shard {shard}: {last_line(shard)[:110]}")
    for shard, (process, log) in enumerate(processes):
        log.close()
        if process.returncode != 0:
            print(Path(f"/content/visual_shard{shard}.log").read_text(errors="ignore")[-3000:])
            raise RuntimeError(f"Shard {shard} lỗi (mã {process.returncode}); chạy lại cell để tiếp tục")
    print("Trích đặc trưng xong:", sum(1 for _ in FEATURE_ROOT.glob("*/*.npz")), "video")

## 8. Train đầu dự đoán
- **Nhãn clip:** đánh nhau nếu ≥ 50% frame của clip là đánh nhau; bình thường nếu không có frame nào và cách đoạn đánh nhau ≥ 1 s; còn lại bỏ qua (nhãn ranh giới không chính xác).
- **Mỗi bước:** `BATCH_SIZE` chuỗi `SEQUENCE_LENGTH` giây; `FIGHT_SHARE` lấy từ video có đánh nhau (80% trong số đó chứa đoạn đánh).
- **Loss:** BCE cân bằng hai lớp + `RANK_WEIGHT` × loss xếp hạng trong từng chuỗi.
- **Validation:** 15% video train, chấm trên video đầy đủ theo frame như tập test. Đầu train in **đường cơ sở năng lượng chuyển động**; mỗi epoch in AUC gộp và AUC trung vị trong từng video đánh nhau. Ngưỡng theo balanced accuracy trên frame.

Luôn chạy với `--resume`: có checkpoint thì train tiếp, lần train đã xong không bị train thêm. Muốn train lại từ đầu, đổi `RUN_NAME`.

In [ ]:
with step('8. Train'):
    run("scripts/train_visual_fight.py", *dataset_args(DATASETS_TRAIN), "--feature-root", FEATURE_ROOT,
        "--output-dir", ARTIFACT_DIR, "--exclude", DEDUP_DIR / "exclude.json",
        "--epochs", EPOCHS, "--steps-per-epoch", STEPS_PER_EPOCH, "--batch-size", BATCH_SIZE,
        "--sequence-length", SEQUENCE_LENGTH, "--fight-share", FIGHT_SHARE, "--rank-weight", RANK_WEIGHT,
        "--select-on", SELECT_ON, "--device", DEVICE, "--resume")

## 9. Báo cáo training

In [ ]:
with step('9. Báo cáo training'):
    import matplotlib.pyplot as plt
    config = json.loads((ARTIFACT_DIR / "config.json").read_text())
    threshold = json.loads((ARTIFACT_DIR / "visual_fight_threshold.json").read_text())
    history = json.loads((ARTIFACT_DIR / "training_history.json").read_text())["history"]
    for key in ["train_videos", "train_fight_videos", "train_fight_clips", "train_normal_clips", "val_videos",
                "parameter_count", "receptive_field_clips", "epochs_completed", "best_epoch",
                "best_val_auc", "best_val_video_auc_median", "val_motion_baseline"]:
        print(f"{key:28s}: {config.get(key)}")
    print("threshold:", threshold)

    epochs = [row["epoch"] for row in history]
    figure, (left, right) = plt.subplots(1, 2, figsize=(13, 4))
    for key in ("train_loss", "train_bce", "train_rank"):
        left.plot(epochs, [r[key] for r in history], label=key)
    right.plot(epochs, [r["val_auc"] for r in history], label="val AUC theo frame")
    right.plot(epochs, [r["val_ap"] for r in history], label="val AP theo frame")
    right.plot(epochs, [r["val_video_auc_median"] for r in history], label="val AUC trong video (trung vị)")
    right.axhline(config["val_motion_baseline"]["video_auc_median"], color="gray", linestyle=":",
                  label="đường cơ sở chuyển động (trong video)")
    for axis in (left, right):
        axis.axvline(config["best_epoch"], color="green", linestyle="--"); axis.grid(alpha=0.3); axis.legend(); axis.set_xlabel("Epoch")
    plt.show()

## 10. Đánh giá trên tập test
Cùng giao thức với luồng B để so trực tiếp:
- UBI-Fights: AUC theo frame trên **toàn bộ** tập test chuẩn; nhóm `_short` (video bình thường chỉ tính 60 s đầu, giống luồng B lần 1–2).
- Nhóm `_crowd`: video trung bình ≥ 5 người/frame, đếm từ khung xương của luồng B (`POSE_ROOT`).
- Surveillance Camera Fight: test chéo theo clip.
- `fight_video_frame_auc_median`: AUC trung vị trong từng video đánh nhau; `motion_baseline_*`: cùng chỉ số cho đường cơ sở năng lượng chuyển động.

Ngưỡng lấy từ validation, không chỉnh theo nhãn test.

In [ ]:
with step('10. Đánh giá'):
    EVAL_DIR = ARTIFACT_DIR / "evaluation"
    crowd = ["--pose-root", POSE_ROOT] if POSE_ROOT.exists() else []
    run("scripts/evaluate_visual_fight.py", *dataset_args(DATASETS_TEST), "--feature-root", FEATURE_ROOT,
        "--model", ARTIFACT_DIR / "visual_fight_head.pt", "--threshold", ARTIFACT_DIR / "visual_fight_threshold.json",
        "--output-dir", EVAL_DIR, "--short-normal-seconds", SHORT_TEST_SECONDS, "--device", DEVICE, *crowd)

## 11. Xem metrics, so với luồng B
Dòng `KẾT LUẬN` cũng được ghi vào `run_log.txt`. Luồng A thực sự học được hành động khi AUC trong video **vượt rõ cả đường cơ sở chuyển động lẫn luồng B** (`fight_v3`: 0,537).

In [ ]:
with step('11. Xem metrics'):
    from IPython.display import Image, display
    evaluation = json.loads((EVAL_DIR / "evaluation.json").read_text())
    print("Video thiếu đặc trưng:", evaluation["missing_feature_caches"], "| nhóm đông người:", evaluation["crowd_groups"])
    for level in ("frame_level", "clip_level"):
        for name, metrics in evaluation[level].items():
            print(f"\n{name} — {level}")
            for key in ["roc_auc", "motion_baseline_roc_auc", "average_precision", "f1", "precision", "recall",
                        "specificity", "accuracy", "balanced_accuracy"]:
                value = metrics.get(key)
                print(f"  {key:24s}: {value if value is None else round(value, 4)}")
            if metrics.get("fight_video_frame_auc_median") is not None:
                print(f"  AUC trong video         : {metrics['fight_video_frame_auc_median']:.4f} "
                      f"(đường cơ sở chuyển động {metrics['motion_baseline_fight_video_frame_auc_median']:.4f}, "
                      f"{metrics['fight_videos']} video đánh nhau)")
            print(f"  confusion: TN={metrics['true_negative']} FP={metrics['false_positive']} FN={metrics['false_negative']} TP={metrics['true_positive']}")

    previous = DRIVE_ROOT / "artifacts" / COMPARE_RUN / "evaluation" / "evaluation.json" if COMPARE_RUN else None
    skeleton = json.loads(previous.read_text()) if previous and previous.exists() else {}
    if skeleton:
        print(f"\nSo với luồng B ({COMPARE_RUN}) trên cùng tập test:")
        for level in ("frame_level", "clip_level"):
            for name, before in skeleton.get(level, {}).items():
                now = evaluation[level].get(name)
                if not now:
                    continue
                for key, label in (("roc_auc", "AUC"), ("fight_video_frame_auc_median", "AUC trong video")):
                    if before.get(key) is not None and now.get(key) is not None:
                        print(f"  {name:16s} {label:16s}: {before[key]:.3f} → {now[key]:.3f} ({now[key] - before[key]:+.3f})")

    ubi = evaluation["frame_level"]["ubi"]
    inside, motion = ubi["fight_video_frame_auc_median"], ubi["motion_baseline_fight_video_frame_auc_median"]
    skeleton_inside = skeleton.get("frame_level", {}).get("ubi", {}).get("fight_video_frame_auc_median")
    note(f"KẾT LUẬN luồng A: UBI AUC theo frame {ubi['roc_auc']:.3f}, AUC trong video {inside:.3f} "
         f"(đường cơ sở chuyển động {motion:.3f}; luồng B {skeleton_inside if skeleton_inside is None else round(skeleton_inside, 3)}); "
         f"SCF theo clip {evaluation['clip_level'].get('scf', {}).get('roc_auc')}")
    for image in sorted(EVAL_DIR.glob("*.png")):
        print(image.name); display(Image(filename=str(image)))

## 12. Đóng gói model
`visual_fight_head.pt` và `visual_fight_threshold.json` đi cùng nhau; giải nén vào thư mục `models/` của repository. Trọng số S3D không nằm trong gói: torchvision tự tải khi chạy lần đầu. Bản ZIP luôn được lưu ở `MyDrive/SENTINEL/artifacts/<RUN_NAME>/`; chỉ mở hộp thoại tải về khi `AUTO_DISCONNECT = False`.

In [ ]:
with step('12. Đóng gói model'):
    import shutil, zipfile
    from google.colab import files
    bundle = Path("/content/sentinel_visual_fight_runtime.zip")
    with zipfile.ZipFile(bundle, "w", zipfile.ZIP_DEFLATED) as archive:
        for name in ["visual_fight_head.pt", "visual_fight_threshold.json"]:
            archive.write(ARTIFACT_DIR / name, arcname=f"models/{name}")
        for name in ["config.json", "split.json", "training_history.json"]:
            archive.write(ARTIFACT_DIR / name, arcname=f"models/metadata/visual_fight_{name}")
        archive.write(EVAL_DIR / "evaluation.json", arcname="models/metadata/visual_fight_evaluation.json")
    shutil.copy2(bundle, ARTIFACT_DIR / bundle.name)
    print(ARTIFACT_DIR / bundle.name, f"{bundle.stat().st_size / 1024**2:.1f} MB")
    if not AUTO_DISCONNECT:
        files.download(str(bundle))

## 13. Kết thúc: ngắt runtime
Khi `AUTO_DISCONNECT = True`, cell này ghi "Hoàn tất" vào `MyDrive/SENTINEL/artifacts/<RUN_NAME>/run_log.txt`, đồng bộ Drive rồi ngắt runtime. Nếu một mục phía trên bị lỗi, runtime cũng tự ngắt và lý do được ghi vào cùng file log.

In [ ]:
finish_and_disconnect(f"Hoàn tất. Kết quả: {ARTIFACT_DIR / 'evaluation' / 'evaluation.json'}")